# LAB 10 - Tokenization và span alignment cho Extractive QA

SQuAD cho answer theo **vị trí ký tự** (`answer_start`) trong context, còn model làm việc với **vị trí token**. Notebook này đi từ ký tự sang token bằng `offset_mapping`, và xử lý context dài bằng sliding window.

In [1]:
import inspect

from qa_model import MODEL_NAME, load_squad, prepare_train_features          # import trước: đặt HF_HOME
import numpy as np
import pandas as pd
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
train = load_squad("train")
print("train:", len(train), "câu hỏi,", train.context.nunique(), "context")

train: 87599 câu hỏi, 18891 context


## 1. Tokenize cặp (question, context)

In [2]:
q, c = "What is the capital of France?", "The capital of France is Paris."
enc = tok(q, c, return_offsets_mapping=True)
print("keys:", list(enc.keys()))
pd.DataFrame({"token": enc.tokens(), "input_id": enc["input_ids"], "attention_mask": enc["attention_mask"],
              "token_type_id": enc["token_type_ids"], "sequence_id": ["-" if s is None else s for s in enc.sequence_ids()],
              "offset": enc["offset_mapping"]}).T

keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
token,[CLS],What,is,the,capital,of,France,?,[SEP],The,capital,of,France,is,Paris,.,[SEP]
input_id,101,1327,1110,1103,2364,1104,1699,136,102,1109,2364,1104,1699,1110,2123,119,102
attention_mask,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
token_type_id,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1
sequence_id,-,0,0,0,0,0,0,0,-,1,1,1,1,1,1,1,-
offset,"(0, 0)","(0, 4)","(5, 7)","(8, 11)","(12, 19)","(20, 22)","(23, 29)","(29, 30)","(0, 0)","(0, 3)","(4, 11)","(12, 14)","(15, 21)","(22, 24)","(25, 30)","(30, 31)","(0, 0)"


- Input có dạng `[CLS] question [SEP] context [SEP]`.
- `offset_mapping`: token chiếm đoạn ký tự `[start, end)` nào trong chuỗi gốc **của nó**. Offset của câu hỏi tính trong câu hỏi, offset của context tính trong context, nên phải dùng `sequence_id` để biết token thuộc phần nào.
- Tokenizer trả về `token_type_ids`, nhưng **DistilBERT không dùng** (forward không có tham số này). Model phân biệt câu hỏi và context chỉ nhờ token `[SEP]`. Vì vậy không thể giả định mọi Transformer đều dùng `token_type_ids`:

In [3]:
model_args = inspect.signature(AutoModelForQuestionAnswering.from_config(
    __import__("transformers").AutoConfig.from_pretrained(MODEL_NAME)).forward).parameters
print("DistilBERT QA forward nhận token_type_ids?", "token_type_ids" in model_args)

DistilBERT QA forward nhận token_type_ids? False


## 2. Character span → token span ("Paris")

In [4]:
a_start = c.index("Paris"); a_end = a_start + len("Paris")
print(f"character span: [{a_start}, {a_end}) → context[{a_start}:{a_end}] = {c[a_start:a_end]!r}")
start_tok = end_tok = None
for t, ((s, e), sid) in enumerate(zip(enc["offset_mapping"], enc.sequence_ids())):
    if sid != 1:
        continue
    if s <= a_start < e:
        start_tok = t
    if s < a_end <= e:
        end_tok = t
print("start token:", start_tok, "| end token:", end_tok, "| decode:", tok.decode(enc["input_ids"][start_tok:end_tok + 1]))
print("Ba loại index của 'Paris': character 25, word", c.split().index("Paris."), "(tính cả dấu chấm dính liền), token", start_tok)

character span: [25, 30) → context[25:30] = 'Paris'
start token: 14 | end token: 14 | decode: Paris
Ba loại index của 'Paris': character 25, word 5 (tính cả dấu chấm dính liền), token 14


## 3. Context dài: sliding window với `stride`

In [5]:
ex = train.loc[train.context.str.len().idxmax()]
f = tok(ex.question, ex.context, truncation="only_second", max_length=384, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True)
print("question:", ex.question, "| answer:", ex.answers[0], "| context:", len(ex.context), "ký tự")
print("keys:", list(f.keys()))
print("overflow_to_sample_mapping:", f["overflow_to_sample_mapping"], "→ mọi window đều thuộc câu hỏi số 0")
feat = prepare_train_features(tok, [ex.question], [ex.context], [ex.starts[0]], [ex.answers[0]])
rows = []
for w in range(len(f["input_ids"])):
    ctx = [o for o, s in zip(f["offset_mapping"][w], f.sequence_ids(w)) if s == 1]
    s, e = feat["start_positions"][w], feat["end_positions"][w]
    rows.append({"window": w, "số token": len(f["input_ids"][w]), "context ký tự": f"{ctx[0][0]} - {ctx[-1][1]}",
                 "start / end": (s, e), "answer trong window": tok.decode(f["input_ids"][w][s:e + 1]) if s else "- ([CLS])"})
pd.DataFrame(rows)

question: What is the largest hottest continuously large area   worldwide? | answer: the Great Desert | context: 3706 ký tự
keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
overflow_to_sample_mapping: [0, 0, 0, 0] → mọi window đều thuộc câu hỏi số 0


,window,số token,context ký tự,start / end,answer trong window
0,0,384,0 - 1642,"(160, 162)",the Great Desert
1,1,384,1085 - 2608,"(0, 0)",- ([CLS])
2,2,384,2056 - 3654,"(0, 0)",- ([CLS])
3,3,153,3069 - 3706,"(0, 0)",- ([CLS])


- Mỗi window là `[CLS] question [SEP] một đoạn context [SEP]`, dài tối đa 384 token.
- `stride=128`: hai window liên tiếp **chồng lên nhau 128 token**, nên answer nằm ở chỗ cắt vẫn xuất hiện trọn vẹn trong ít nhất một window.
- `overflow_to_sample_mapping` cho biết window thuộc câu hỏi nào, dùng để gom kết quả các window khi dự đoán.
- Window không chứa trọn answer được gán start = end = 0 (token `[CLS]`), nghĩa là "không có answer ở đây".

## 4. Bốn trường hợp đặc biệt trên dữ liệu thật

In [6]:
def offsets_ctx(question, context, **kw):
    e = tok(question, context, return_offsets_mapping=True, **kw)
    return [o for o, s in zip(e["offset_mapping"], e.sequence_ids()) if s == 1]

sample = train.sample(20000, random_state=0)
mid_start, mid_end = [], []
for i, r in sample.iterrows():
    offs = offsets_ctx(r.question, r.context, truncation="only_second", max_length=512)
    a, b = r.starts[0], r.starts[0] + len(r.answers[0])
    if any(s < a < e for s, e in offs):
        mid_start.append(i)
    if any(s < b < e for s, e in offs):
        mid_end.append(i)
print(f"answer bắt đầu giữa token: {len(mid_start)} / {len(sample)} ({100 * len(mid_start) / len(sample):.2f}%)")
print(f"answer kết thúc giữa token: {len(mid_end)} / {len(sample)} ({100 * len(mid_end) / len(sample):.2f}%)")
for name, idx in [("bắt đầu giữa token", mid_start[:3]), ("kết thúc giữa token", mid_end[:3])]:
    for i in idx:
        r = train.loc[i]
        f1 = prepare_train_features(tok, [r.question], [r.context], [r.starts[0]], [r.answers[0]], max_length=512)
        k = next(k for k, s in enumerate(f1["start_positions"]) if s)
        s, e = f1["start_positions"][k], f1["end_positions"][k]
        a = r.starts[0]
        print(f"  [{name}] gold {r.answers[0]!r} | quanh đó: {r.context[max(0, a - 15):a + len(r.answers[0]) + 15]!r} "
              f"| token span → {tok.decode(f1['input_ids'][k][s:e + 1])!r}")

answer bắt đầu giữa token: 28 / 20000 (0.14%)
answer kết thúc giữa token: 41 / 20000 (0.20%)
  [bắt đầu giữa token] gold 'n March 2006, the foundation announced a US$5 million grant for the International Justice Mission (IJM)' | quanh đó: 'In March 2006, the foundation announced a US$5 million grant for the International Justice Mission (IJM), a human right' | token span → 'In March 2006, the foundation announced a US $ 5 million grant for the International Justice Mission ( IJM )'
  [bắt đầu giữa token] gold 'yptian Se' | quanh đó: 'as the Greco-Egyptian Serapis, eastern ' | token span → 'Egyptian Ser'
  [bắt đầu giữa token] gold 'he word slovo ("word") and the related slava ("fame") and slukh ("hearing")' | quanh đó: 'The word slovo ("word") and the related slava ("fame") and slukh ("hearing") originate from' | token span → 'The word slovo ( " word " ) and the related slava ( " fame " ) and slukh ( " hearing " )'
  [kết thúc giữa token] gold '3' | quanh đó: 'tasy at number 353.' | to

Khi answer bắt đầu hoặc kết thúc giữa token, label phải lấy **cả token**, nên span khôi phục được rộng hơn gold. Trên SQuAD trường hợp này rất hiếm (0.1 - 0.2%), và phần lớn là lỗi annotation: gold bị cắt giữa chữ, ví dụ "n March 2006" hay "yptian Se".

In [7]:
rows = []
for L, S in [(128, 42), (256, 85), (384, 128)]:
    f = prepare_train_features(tok, sample.question, sample.context, [s[0] for s in sample.starts],
                               [a[0] for a in sample.answers], max_length=L, stride=S)
    n_win = np.bincount(f["sample_idx"], minlength=len(sample))
    has = np.zeros(len(sample), bool)
    for s_idx, sp in zip(f["sample_idx"], f["start_positions"]):
        has[s_idx] |= sp > 0
    rows.append({"max_length": L, "stride": S, "window / câu hỏi": n_win.mean(), "% câu hỏi cần > 1 window": 100 * (n_win > 1).mean(),
                 "% window không chứa answer ([CLS])": 100 * np.mean(np.array(f["start_positions"]) == 0),
                 "% câu hỏi có answer trong ít nhất 1 window": 100 * has.mean()})
    if L == 384:
        # khôi phục answer: token span → offset → đoạn ký tự trong context, so với gold
        enc = tok(list(sample.question), list(sample.context), truncation="only_second", max_length=L, stride=S,
                  return_overflowing_tokens=True, return_offsets_mapping=True)
        ok = []
        for k, (s, e) in enumerate(zip(f["start_positions"], f["end_positions"])):
            if s:
                r = sample.iloc[f["sample_idx"][k]]
                ok.append(r.context[enc["offset_mapping"][k][s][0]:enc["offset_mapping"][k][e][1]] == r.answers[0])
        print(f"max_length 384: token span → ký tự khôi phục đúng nguyên văn gold ở {100 * np.mean(ok):.2f}% window có answer")
pd.DataFrame(rows).round(2)

max_length 384: token span → ký tự khôi phục đúng nguyên văn gold ở 99.68% window có answer


,max_length,stride,window / câu hỏi,% câu hỏi cần > 1 window,% window không chứa answer ([CLS]),% câu hỏi có answer trong ít nhất 1 window
0,128,42,2.21,78.96,45.43,100.0
1,256,85,1.12,11.08,8.54,100.0
2,384,128,1.01,1.23,1.07,100.0


**Nhận xét**

- Offset mapping đổi được character span sang token span: khôi phục đúng nguyên văn gold ở 99.68% window có answer. Phần còn lại chủ yếu là gold bị cắt giữa chữ do lỗi annotation.
- Với max_length 384, chỉ 1.2% câu hỏi cần nhiều hơn 1 window. Với 128 thì 79% câu hỏi phải chia window và 45% window không chứa answer (nhãn [CLS]), nên model phải học thêm việc nhận ra "window này không có đáp án".
- Nhờ stride, ở cả ba độ dài, 100% câu hỏi đều có answer nằm trọn trong ít nhất một window.
- DistilBERT không dùng `token_type_ids`, nên chỉ nhờ `[SEP]` để phân biệt câu hỏi và context.